# AlphaFlow / ESMFlow on Google Colab

Generate **protein conformational ensembles** with [AlphaFlow](https://github.com/bjing2016/alphaflow) — a fine-tuning of AlphaFold (and ESMFold) with a flow-matching objective from the ICML 2024 paper [*AlphaFold Meets Flow Matching for Generating Protein Ensembles*](https://arxiv.org/abs/2402.04845) (Jing, Berger, Jaakkola).

This notebook gives you the full set of AlphaFlow functionalities on Colab:

| Capability | Cell |
|---|---|
| Install AlphaFlow + OpenFold + dependencies | §1 |
| Pick a model (PDB / MD / MD+Templates, base / distilled / 12L) | §2 |
| Provide one or many sequences (text or CSV upload) | §3 |
| Auto-generate MSAs via the ColabFold MMseqs2 server | §4 |
| Provide reference templates for `MD+Templates` models | §5 |
| Sample an ensemble with full control over `samples`, `steps`, `tmax`, `self_cond`, `resample`, `subsample`, `noisy_first`, `no_diffusion` | §6 |
| Visualise the ensemble in 3D (animated) | §7 |
| Quantitative analysis — pairwise RMSD, per-residue RMSF, PCA of the conformational landscape | §8 |
| Download the ensemble PDB files | §9 |

### Before you start
1. Switch to a GPU runtime: **Runtime → Change runtime type → T4** (or better, A100 / L4).
2. Run the cells in order. Cells with form fields (the gear ⚙️ icon to the right) let you change parameters without editing code.
3. The **install step requires a one-time kernel restart** (handled automatically by `condacolab`).

---
## §1 — Install AlphaFlow

AlphaFlow depends on **OpenFold**, which builds custom CUDA kernels and is pinned to **Python 3.9 + CUDA 11.8 + PyTorch 1.13**. We use [`condacolab`](https://github.com/conda-incubator/condacolab) to set up a clean conda environment matching those constraints.

**Install runs in two stages**:
1. **§1.1** — Bootstrap conda. The kernel restarts automatically (this is normal — just continue with §1.2 once it comes back).
2. **§1.2** — Install CUDA 11.8, PyTorch, OpenFold, and clone the AlphaFlow repository. Takes **~10–15 minutes**.

In [ ]:
#@title §1.0 — Check GPU
#@markdown Verify a GPU is attached. If you see *No GPU detected*, switch the runtime type before continuing.
import subprocess
try:
    out = subprocess.check_output(['nvidia-smi'], text=True)
    print(out)
except Exception:
    print('No GPU detected. Use Runtime -> Change runtime type -> GPU.')

In [ ]:
#@title §1.1 — Bootstrap conda (kernel will auto-restart)
#@markdown After the restart message appears, jump straight to §1.2 — do **not** rerun this cell.
!pip install -q condacolab
import condacolab
condacolab.install_miniforge()

In [ ]:
#@title §1.2 — Install CUDA 11.8, PyTorch, OpenFold, AlphaFlow (~10–15 min)
#@markdown This installs everything inside the conda base environment and clones the AlphaFlow repository to `/content/alphaflow`.
import condacolab, os, sys
condacolab.check()

# 1. CUDA 11.8 toolkit (runtime + dev headers needed by OpenFold's kernels).
#    We keep the Python that condacolab installed — all pinned packages are
#    compatible with it and downgrading mid-kernel would not take effect.
!conda install -y -q -c nvidia/label/cuda-11.8.0 cuda cuda-cudart-dev libcusparse-dev libcusolver-dev libcublas-dev
!ln -sf $CONDA_PREFIX/lib/libcudart_static.a $CONDA_PREFIX/lib/libcudart.a

# 2. Pinned Python dependencies (matches the AlphaFlow README).
!pip install -q numpy==1.21.2 pandas==1.5.3
!pip install -q torch==1.12.1+cu113 -f https://download.pytorch.org/whl/torch_stable.html
!pip install -q biopython==1.79 dm-tree==0.1.6 modelcif==0.7 ml-collections==0.1.0 scipy==1.7.1 absl-py einops
!pip install -q pytorch_lightning==2.0.4 fair-esm mdtraj==1.9.9 wandb

# 3. OpenFold (custom CUDA extensions; needs CUDA_HOME pointing at the conda env).
os.environ['CUDA_HOME'] = os.environ['CONDA_PREFIX']
!CUDA_HOME=$CONDA_PREFIX pip install -q 'openfold @ git+https://github.com/aqlaboratory/openfold.git@103d037'

# 4. Helpers used later in this notebook (visualisation + plotting).
!pip install -q py3Dmol matplotlib scikit-learn

# 5. Clone AlphaFlow.
if not os.path.isdir('/content/alphaflow'):
    !git clone -q https://github.com/bjing2016/alphaflow.git /content/alphaflow
%cd /content/alphaflow
if '/content/alphaflow' not in sys.path:
    sys.path.insert(0, '/content/alphaflow')

print('\n✅ Installation complete.')

---
## §2 — Choose a model and download weights

AlphaFlow ships in three flavours — pick the one that matches what you want to model:

| Model | What it models | Needs MSA? | Needs template? |
|---|---|---|---|
| **PDB** | Alternative experimental conformations (X-ray / cryo-EM) | yes (AlphaFlow) / no (ESMFlow) | no |
| **MD** | All-atom MD ensemble at 300 K | yes / no | no |
| **MD+Templates** | Same as MD, but conditioned on a known structure | yes / no | **yes** |

Each model has a **base** version (slower, more accurate) and a **distilled** version (faster). For `MD+Templates` there is also a **12L** version (12-layer Evoformer, 2.5× faster, small accuracy loss). `ESMFlow` is the corresponding ESMFold-based fine-tune and skips the MSA stage entirely.

If you pick a **distilled** model the inference cell (§6) will automatically set `--noisy_first --no_diffusion`. For the **PDB** model `--self_cond --resample` are recommended.

In [ ]:
#@title §2.1 — Pick the model
model_family = 'AlphaFlow'  #@param ['AlphaFlow', 'ESMFlow']
training_set = 'MD+Templates'  #@param ['PDB', 'MD', 'MD+Templates']
version = 'base'  #@param ['base', 'distilled', '12l-base', '12l-distilled']

#@markdown The 12L variants only exist for AlphaFlow MD+Templates.

_VALID = {
    ('AlphaFlow', 'PDB'):           {'base': 'alphaflow_pdb_base_202402.pt',
                                      'distilled': 'alphaflow_pdb_distilled_202402.pt'},
    ('AlphaFlow', 'MD'):            {'base': 'alphaflow_md_base_202402.pt',
                                      'distilled': 'alphaflow_md_distilled_202402.pt'},
    ('AlphaFlow', 'MD+Templates'):  {'base': 'alphaflow_md_templates_base_202402.pt',
                                      'distilled': 'alphaflow_md_templates_distilled_202402.pt',
                                      '12l-base': 'alphaflow_12l_md_templates_base_202406.pt',
                                      '12l-distilled': 'alphaflow_12l_md_templates_distilled_202406.pt'},
    ('ESMFlow',   'PDB'):           {'base': 'esmflow_pdb_base_202402.pt',
                                      'distilled': 'esmflow_pdb_distilled_202402.pt'},
    ('ESMFlow',   'MD'):            {'base': 'esmflow_md_base_202402.pt',
                                      'distilled': 'esmflow_md_distilled_202402.pt'},
    ('ESMFlow',   'MD+Templates'):  {'base': 'esmflow_md_templates_base_202402.pt',
                                      'distilled': 'esmflow_md_templates_distilled_202402.pt'},
}

key = (model_family, training_set)
if key not in _VALID or version not in _VALID[key]:
    raise ValueError(f'Combination {model_family}/{training_set}/{version} is not available — see the table in §2.')

weight_filename = _VALID[key][version]
weights_url = f'https://huggingface.co/bjing-mit/alphaflow/resolve/main/params/{weight_filename}'
MODE = 'alphafold' if model_family == 'AlphaFlow' else 'esmfold'
USE_TEMPLATES = (training_set == 'MD+Templates')
IS_DISTILLED = ('distilled' in version)
IS_PDB = (training_set == 'PDB')

print(f'Mode               : {MODE}')
print(f'Templates required : {USE_TEMPLATES}')
print(f'Distilled          : {IS_DISTILLED}')
print(f'Weights file       : {weight_filename}')

In [ ]:
#@title §2.2 — Download the weights
import os
os.makedirs('/content/alphaflow/params', exist_ok=True)
WEIGHTS_PATH = f'/content/alphaflow/params/{weight_filename}'
if not os.path.exists(WEIGHTS_PATH):
    !wget -q --show-progress -O {WEIGHTS_PATH} {weights_url}
else:
    print('Weights already present — skipping download.')
print('Weights at:', WEIGHTS_PATH)
!ls -lh {WEIGHTS_PATH}

---
## §3 — Provide your input sequences

AlphaFlow expects a CSV with at least two columns:
- **`name`** — a unique identifier (must be filename-safe; this is what output files are named after).
- **`seqres`** — the amino-acid sequence (single-letter codes, no gaps).

You have two options:
1. **Type sequences inline** in the form below — quickest for one or two proteins.
2. **Upload your own CSV** — useful for batch runs.

Examples shipped with the repo: `splits/atlas_test.csv`, `splits/cameo2022.csv`, `splits/pdb_test.csv`.

In [ ]:
#@title §3 — Define your input
input_mode = 'inline'  #@param ['inline', 'upload_csv', 'use_repo_split']

#@markdown **inline** — define `name`/`seqres` pairs in the boxes below.
name_1 = 'my_protein'  #@param {type: 'string'}
seqres_1 = 'MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLA'  #@param {type: 'string'}
#@markdown Add as many extra entries as you want, comma-separated. Leave blank to skip.
additional_names = ''  #@param {type: 'string'}
additional_seqres = ''  #@param {type: 'string'}

#@markdown **upload_csv** — when selected, a file picker opens so you can upload your CSV.
#@markdown
#@markdown **use_repo_split** — point to one of the CSVs already in the repo.
repo_split = 'splits/atlas_test.csv'  #@param ['splits/atlas_test.csv', 'splits/cameo2022.csv', 'splits/pdb_test.csv']

import os, pandas as pd
INPUT_CSV = '/content/alphaflow/input.csv'

if input_mode == 'inline':
    rows = [{'name': name_1, 'seqres': seqres_1.replace(' ', '').upper()}]
    extra_n = [n.strip() for n in additional_names.split(',') if n.strip()]
    extra_s = [s.strip().replace(' ', '').upper() for s in additional_seqres.split(',') if s.strip()]
    if len(extra_n) != len(extra_s):
        raise ValueError('additional_names and additional_seqres must have the same length.')
    for n, s in zip(extra_n, extra_s):
        rows.append({'name': n, 'seqres': s})
    df = pd.DataFrame(rows)
    df.to_csv(INPUT_CSV, index=False)
elif input_mode == 'upload_csv':
    from google.colab import files
    print('Select your CSV (must contain `name` and `seqres` columns):')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file uploaded.')
    src = list(uploaded.keys())[0]
    df = pd.read_csv(src)
    if not {'name', 'seqres'}.issubset(df.columns):
        raise ValueError('CSV must contain `name` and `seqres` columns.')
    df.to_csv(INPUT_CSV, index=False)
elif input_mode == 'use_repo_split':
    df = pd.read_csv(f'/content/alphaflow/{repo_split}')
    df.to_csv(INPUT_CSV, index=False)

print(f'Wrote {len(df)} sequence(s) to {INPUT_CSV}:')
df[['name', 'seqres']].assign(seqlen=df.seqres.str.len()).head(20)

---
## §4 — Generate MSAs (AlphaFlow only)

**AlphaFlow** needs an MSA per input sequence, supplied as `{msa_dir}/{name}/a3m/{name}.a3m`. The cell below queries the ColabFold MMseqs2 server (`https://api.colabfold.com`) just like the official `scripts/mmseqs_query.py`.

If you picked **ESMFlow** in §2 you can skip this cell — ESMFlow uses the ESM-2 single-sequence representation, no MSA required.

In [ ]:
#@title §4 — Query the ColabFold MMseqs2 server
MSA_DIR = '/content/alphaflow/alignment_dir'

if MODE == 'esmfold':
    print('ESMFlow selected — skipping MSA generation.')
else:
    import os
    os.makedirs(MSA_DIR, exist_ok=True)
    !cd /content/alphaflow && python -m scripts.mmseqs_query --split {INPUT_CSV} --outdir {MSA_DIR}
    print('\nMSA layout:')
    !ls {MSA_DIR}

---
## §5 — Provide template structures (`MD+Templates` only)

If you selected an **`MD+Templates`** model, AlphaFlow needs a reference PDB file per input sequence, with filename `{templates_dir}/{name}.pdb`. The PDB must contain a **single chain** with **no residue gaps**, matching the corresponding sequence in §3.

Common sources for the template structure:
- A PDB file from RCSB (cleaned to one chain).
- An AlphaFold prediction.
- A previously generated AlphaFlow sample.

Skip this cell if your model is **PDB** or **MD**.

In [ ]:
#@title §5 — Upload templates (skip if not using `MD+Templates`)
TEMPLATES_DIR = '/content/alphaflow/templates_dir'

if not USE_TEMPLATES:
    print('Selected model does not need templates — skipping.')
else:
    import os
    os.makedirs(TEMPLATES_DIR, exist_ok=True)
    from google.colab import files
    print(f'Upload one PDB per sequence, named exactly <name>.pdb (where <name> matches column `name` in §3).')
    uploaded = files.upload()
    for fn, content in uploaded.items():
        with open(os.path.join(TEMPLATES_DIR, fn), 'wb') as f:
            f.write(content)
    print('\nTemplates directory:')
    !ls {TEMPLATES_DIR}

---
## §6 — Run inference

All command-line flags exposed by `predict.py` are available below. Defaults follow the recommendations in the README:

| Flag | Meaning |
|---|---|
| `samples` | Number of conformations to draw per sequence (each is one ODE trajectory). |
| `steps` | Number of integration steps. Default is 10. |
| `tmax` | Truncated start time (default 1.0). Lower → less diversity, higher precision. |
| `self_cond` | Recommended for the **PDB** model. |
| `resample` | Re-sample the MSA between draws (recommended for **PDB**, useful for MSA-subsampling AF). |
| `subsample` | If set, sub-sample the MSA to this depth — see *Wayment-Steele et al., eLife 2022*. |
| `noisy_first` + `no_diffusion` | Required for **distilled** models — predict in a single step from noise. |

Sensible defaults are pre-selected based on §2.

In [ ]:
#@title §6 — Inference parameters
samples = 5  #@param {type: 'integer'}
steps = 10  #@param {type: 'integer'}
tmax = 1.0  #@param {type: 'number'}
self_cond = False  #@param {type: 'boolean'}
resample = False  #@param {type: 'boolean'}
subsample = 0  #@param {type: 'integer'}
noisy_first = False  #@param {type: 'boolean'}
no_diffusion = False  #@param {type: 'boolean'}
#@markdown Set this to a comma-separated subset of `name`s to run inference only on those rows. Leave blank to run on all.
pdb_id_filter = ''  #@param {type: 'string'}

import os
OUTPDB = '/content/alphaflow/outpdb'
os.makedirs(OUTPDB, exist_ok=True)

# Apply README-recommended flags automatically.
if IS_DISTILLED:
    noisy_first = True
    no_diffusion = True
    print('[auto] distilled model → enabling --noisy_first --no_diffusion')
if IS_PDB:
    self_cond = True
    resample = True
    print('[auto] PDB model → enabling --self_cond --resample')

cmd = [
    'python', 'predict.py',
    '--mode', MODE,
    '--input_csv', INPUT_CSV,
    '--weights', WEIGHTS_PATH,
    '--samples', str(samples),
    '--steps', str(steps),
    '--tmax', str(tmax),
    '--outpdb', OUTPDB,
    '--runtime_json', f'{OUTPDB}/runtime.json',
]
if MODE == 'alphafold':
    cmd += ['--msa_dir', MSA_DIR]
if USE_TEMPLATES:
    cmd += ['--templates_dir', TEMPLATES_DIR]
if self_cond:    cmd += ['--self_cond']
if resample:     cmd += ['--resample']
if noisy_first:  cmd += ['--noisy_first']
if no_diffusion: cmd += ['--no_diffusion']
if subsample:    cmd += ['--subsample', str(subsample)]
if pdb_id_filter.strip():
    cmd += ['--pdb_id'] + [x.strip() for x in pdb_id_filter.split(',') if x.strip()]

print('Running:\n  ' + ' '.join(cmd) + '\n')
import subprocess
subprocess.run(cmd, cwd='/content/alphaflow', check=True)
print('\nGenerated PDBs:')
!ls -lh {OUTPDB}

---
## §7 — Visualise the ensemble

Each output PDB is a multi-model file (one `MODEL ... ENDMDL` block per sample). The cell below uses **py3Dmol** to animate through the ensemble, colouring each conformer by its B-factor (which AlphaFold uses to encode the predicted lDDT).

In [ ]:
#@title §7 — 3D visualisation
target_name = 'my_protein'  #@param {type: 'string'}
show_animation = True  #@param {type: 'boolean'}
colour_by = 'spectrum'  #@param ['spectrum', 'bfactor', 'chain']

import os, py3Dmol
pdb_path = f'{OUTPDB}/{target_name}.pdb'
if not os.path.exists(pdb_path):
    raise FileNotFoundError(f'No ensemble found for `{target_name}`. Available: {os.listdir(OUTPDB)}')

with open(pdb_path) as f:
    pdb_str = f.read()

view = py3Dmol.view(width=720, height=520)
view.addModelsAsFrames(pdb_str, 'pdb') if show_animation else view.addModel(pdb_str, 'pdb')
if colour_by == 'spectrum':
    view.setStyle({'cartoon': {'color': 'spectrum'}})
elif colour_by == 'bfactor':
    view.setStyle({'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'roygb', 'min': 50, 'max': 90}}})
else:
    view.setStyle({'cartoon': {'colorscheme': 'chain'}})
view.zoomTo()
if show_animation:
    view.animate({'loop': 'forward', 'interval': 500})
view.show()

---
## §8 — Quantitative ensemble analysis (optional)

These analyses mirror what `scripts/analyze_ensembles.py` reports against MD reference data. Without an MD reference, we report:
- **Pairwise C-α RMSD distribution** — measures the conformational diversity within the ensemble.
- **Per-residue C-α RMSF** — flexibility of each residue across the ensemble.
- **PCA of C-α coordinates** — shows the dominant collective motions captured.

If you also have an MD trajectory (e.g. an ATLAS target), you can run the full evaluation pipeline using `scripts/analyze_ensembles.py` followed by `scripts/print_analysis.py` (see §10 below).

In [ ]:
#@title §8 — Pairwise RMSD, RMSF, PCA
target_name = 'my_protein'  #@param {type: 'string'}

import os, mdtraj as md, numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA

pdb_path = f'{OUTPDB}/{target_name}.pdb'
traj = md.load(pdb_path)
ca = traj.atom_slice([a.index for a in traj.top.atoms if a.name == 'CA'])
ca.superpose(ca, frame=0)
n = ca.n_frames
print(f'{n} conformers, {ca.n_atoms} C-alphas')

# Pairwise RMSD matrix.
rmsd_mat = np.zeros((n, n))
for i in range(n):
    rmsd_mat[i] = md.rmsd(ca, ca, frame=i) * 10  # nm -> Å

# Per-residue RMSF (Å).
rmsf = md.rmsf(ca, ca, frame=0) * 10

# PCA on C-α coordinates.
xyz = ca.xyz.reshape(n, -1)
pca = PCA(n_components=min(3, n))
coords = pca.fit_transform(xyz)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
im = ax[0].imshow(rmsd_mat, cmap='viridis')
ax[0].set_title(f'Pairwise C-α RMSD (Å)\nmean={rmsd_mat[np.triu_indices(n, 1)].mean():.2f}')
ax[0].set_xlabel('sample'); ax[0].set_ylabel('sample')
fig.colorbar(im, ax=ax[0], shrink=0.8)

ax[1].plot(rmsf)
ax[1].set_title('Per-residue C-α RMSF (Å)')
ax[1].set_xlabel('residue index'); ax[1].set_ylabel('RMSF')
ax[1].grid(alpha=0.3)

if coords.shape[1] >= 2:
    ax[2].scatter(coords[:, 0], coords[:, 1], c=np.arange(n), cmap='plasma')
    ax[2].set_title(f'PCA — PC1 vs PC2\n(var explained: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%)')
    ax[2].set_xlabel('PC1'); ax[2].set_ylabel('PC2')
    ax[2].grid(alpha=0.3)
else:
    ax[2].text(0.5, 0.5, 'Need >=2 samples for PCA', ha='center', transform=ax[2].transAxes)
plt.tight_layout(); plt.show()

print(f'\nMean pairwise C-α RMSD : {rmsd_mat[np.triu_indices(n, 1)].mean():.2f} Å')
print(f'Median per-residue RMSF: {np.median(rmsf):.2f} Å')
print(f'Top-3 PC variance      : {pca.explained_variance_ratio_[:3] * 100}')

---
## §9 — Download the ensembles

Bundle every PDB in `outpdb/` into a single zip and trigger a browser download.

In [ ]:
#@title §9 — Zip and download
import shutil
ZIP_BASE = '/content/alphaflow_outputs'
shutil.make_archive(ZIP_BASE, 'zip', OUTPDB)
from google.colab import files
files.download(ZIP_BASE + '.zip')

---
## §10 — (Optional) Full ATLAS evaluation pipeline

AlphaFlow ships with two extra scripts to reproduce the paper's MD-vs-ensemble evaluation:

```bash
# 1. Download ATLAS reference trajectories (each ~ a few hundred MB).
bash scripts/download_atlas.sh

# 2. Compute analysis pickle (RMSF, EMD, PCA, contact maps, SASA, MI, ...).
python -m scripts.analyze_ensembles \\
    --atlas_dir <ATLAS_DIR> \\
    --pdbdir    <ENSEMBLE_DIR> \\
    --num_workers 4

# 3. Pretty-print a comparison table from one or more analysis pickles.
python -m scripts.print_analysis <ENSEMBLE_DIR>/out.pkl
```

ATLAS is large — running this in Colab works but you should mount Google Drive (`from google.colab import drive; drive.mount('/content/drive')`) and point `--atlas_dir` at a Drive folder so you don't lose data when the runtime resets.

### Pre-computed ensembles
If you just want the ensembles from the paper, every model has a downloadable `.zip` of its samples:
```
https://huggingface.co/bjing-mit/alphaflow/resolve/main/samples/<weight_filename without .pt>.zip
```
(see the **Ensembles** section of the README for the full list).

### Training
Training (PDB or ATLAS, base or distilled, with or without templates) is exposed by `train.py` and follows the recipes in the README. It is data- and GPU-heavy and is generally not feasible inside Colab; use a multi-GPU node instead.

---
*Notebook by the AlphaFlow community. If you use AlphaFlow please cite Jing et al., ICML 2024.*